# 03c · Gerçek NAS: sıralama referansı ve finalist seçimi (`nas_rank1`)

**Ne yapar:** 03b'nin önceden kayıtlı kuralla seçtiği **en çok 11 mimariyi** (≤ 3 Pareto + 7 gecikmeye göre katmanlı + Nano@416) **sıfırdan 60 epoch** eğitir (ön eğitimli ağırlık yok; süper ağdan ağırlık yok) → (1) süper ağ sıralamasının bağımsız eğitimle uyumu (Kendall τ), (2) Karar 6 finalist kuralı.

**Bu bir kısmi eğitim referansıdır** (Karar 6): 300 epoch sıralamasını garanti etmez; τ seçilmiş 11'lik panele koşullu, bootstrap birimi mimari (grup yapısı korunur), tohum belirsizliği kapsanmaz.

**Tarif (tüm adaylara birebir aynı; manifestte):** YOLOX-Nano sıfırdan tarifi — tam train2017, batch 64, fp16, SGD 0,9 nesterov, lr 0,01/64 görsel, yoloxwarmcos, min_lr_ratio 0,05, weight decay 5e-4, EMA, mosaic 0,5, mixup kapalı, çok ölçek 320–640 — **60 epoch, ısınma 1, son no_aug 3 (+1) epoch** — YOLOX temel tarifinin 5/300 ve 15/300 oranlarıyla **orantılı kısaltılmış reçete** (Furkan, 2026-10-17); 300 epoch ile aynı sıralamayı sağladığı iddia edilmez. Başlatma tohumu 0, veri/artırma tohumu 0 **tüm adaylarda ortak** (aynı veri sırası ve artırma: karşılaştırma gürültüsü azalır).

**Değerlendirme:** son epoch EMA ağırlığı, **val2017-A (2500)**, kendi çözünürlüğünde, fp32, conf 0,001, NMS 0,65. B mühürlü.

**Finalist kuralı (Karar 6, önceden kayıtlı):** AP_A,60(aday) ≥ AP_A,60(Nano@416) − 1,0 (0–100, mutlak) **ve** asgari gecikme azalması T_aday ≤ (1 − δ) · T_N, δ = max(0,01; 3 · CV_N), CV_N Nano'nun ayrı benchmark tekrarlarının medyanlarından (aday sonuçları görülmeden; pilotla belirlenen mühendislik güvenlik payı, istatistiksel anlamlılık değil) → en düşük medyan gecikme. Adayların kendi tekrar CV'leri raporlanır. Uygun aday yoksa eşik gevşetilmez. Cihaz ölçümü Karar 3 protokolü (100 ısınma + 60 sn × 3, ORT CPU 4 thread), `scripts/n4_latency.py`.

**Kesintiye dayanıklılık:** her koşu kendi doğrulanmış deposunda (`runs/<aday>/ckpt/`), 300 iterasyonda bir + epoch sonu; hücre 9 yeniden çalıştırılınca bitmiş koşular atlanır, yarım koşu en yeni doğrulanmış kayıttan devam eder. Hücre 7: GPU'da sıfırdan trainer kesinti testi (kayıt yazılırken öldür + kill -9, izler birebir).

**Süre/maliyet (tahmin, hücre 7 ölçer):** tek ağ epoch ≈ 886 sn (nas_r2) → 60 ep ≈ 14,8 sa/aday (büyük kernel ≈ +%5–10) → 11 koşu ≈ 165–175 sa ≈ 255–270 CU + değerlendirme ≈ 11 × 1 dk.

**Kayıtlar** (`Drive edge_ai/results/nas_rank1/`): `manifest.json` · `resume_test.json` · `runs/<aday>/{status.json, train_log.txt, eval_A_log.txt, ckpt/}` · `results.csv` · `rank_analysis.json` · `latency_pkg/` (Jetson paketi) · `speed_margin.json` · `finalist.json`.

Kod: `exps/nas/scratch_subnet.py`, `scripts/{rank_select, nas_data, ckpt_store, nas_export, n4_latency}.py`.

In [ ]:
# 1 · Ortam ve sürüm kaydı
import json, platform, subprocess, sys, datetime, os

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
    except Exception as e:
        return f"ERR {e}"

env = {
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu": sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") or "none",
    "cuda_nvcc": sh("nvcc --version | tail -1"),
    "cpu_count": os.cpu_count(),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1),
    "disk_free_content": sh("df -h /content | tail -1"),
}
try:
    import torch, torchvision
    env.update(torch=torch.__version__, torchvision=torchvision.__version__,
               torch_cuda=torch.version.cuda, cuda_available=torch.cuda.is_available())
except ImportError:
    env["torch"] = "not installed"
print(json.dumps(env, indent=2))

In [ ]:
# 2 · Kod: depo + YOLOX fork sabit commit (exp ve betik depoda: exps/nas, scripts)
REPO_URL = "https://github.com/Bey3fendi/edge-ai-object-detection.git"
YOLOX_COMMIT = "3ec6263"
WS = "/content/edge_ai_workspace"

import os, subprocess
if not os.path.isdir(WS):
    subprocess.run(["git", "clone", "--recurse-submodules", REPO_URL, WS], check=True)
else:  # runtime aynı kaldıysa depodaki son exp/betik sürümünü al
    subprocess.run(["git", "-C", WS, "pull", "-q", "--ff-only"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "fetch", "--all", "-q"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "checkout", "-q", YOLOX_COMMIT], check=True)

env["repo_commit"] = sh(f"git -C {WS} rev-parse --short HEAD")
env["yolox_commit"] = sh(f"git -C {WS}/YOLOX rev-parse --short HEAD")
env["yolox_dirty"] = bool(sh(f"git -C {WS}/YOLOX status --porcelain"))
print({k: env[k] for k in ("repo_commit", "yolox_commit", "yolox_dirty")})
assert env["yolox_commit"].startswith(YOLOX_COMMIT)
env["repo_dirty"] = bool(sh(f"git -C {WS} status --porcelain"))
assert not env["repo_dirty"], "depo kirli: kayıtlı commit koşulan kodu temsil etmez"

In [ ]:
# 3 · YOLOX bağımlılıkları
# Not: `pip install -e .` (setup.py develop) güncel setuptools + py3.13'te kırılıyor; YOLOX saf Python
# olduğu için kurulum yerine PYTHONPATH kullanılır. onnx-simplifier eğitimde gerekmez → atlandı.
# onnx + onnxruntime: check_supernet.py'nin ONNX opset 13 eşdeğerlik denetimi için (hücre 7).
!pip install -q loguru tabulate thop ninja pycocotools psutil tensorboard opencv-python-headless onnx onnxruntime

import sys, os
YOLOX_DIR = f"{WS}/YOLOX"
if YOLOX_DIR not in sys.path:
    sys.path.insert(0, YOLOX_DIR)
os.environ["PYTHONPATH"] = YOLOX_DIR + ":" + os.environ.get("PYTHONPATH", "")

import importlib.metadata as md
for p in ["torch", "torchvision", "numpy", "opencv-python-headless", "pycocotools", "loguru", "thop", "onnx", "onnxruntime"]:
    try:
        env[f"pkg_{p}"] = md.version(p)
    except md.PackageNotFoundError:
        env[f"pkg_{p}"] = None
import yolox
from yolox.exp import get_exp
exp = get_exp(None, "yolox-nano")
print("yolox import OK:", yolox.__file__)
print({k: v for k, v in env.items() if k.startswith("pkg_")})
SCRIPTS = f"{WS}/scripts"
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)
NAS_EXPS = f"{WS}/exps/nas"
if NAS_EXPS not in sys.path:
    sys.path.insert(0, NAS_EXPS)
import importlib, nas_search as ns, nas_data as nd, supernet as sn, supernet_search as ss
for _m in (ns, nd, sn, ss):
    importlib.reload(_m)
EXP_FILE = f"{WS}/exps/nas/scratch_subnet.py"
import rank_select as rs, nas_export as nx
importlib.reload(rs); importlib.reload(nx)

In [ ]:
# 4 · Drive + ağırlık (SHA256)
# İlk mount tarayıcıda kullanıcı onayı ister; bağlıysa tekrar denemez.
env.pop("pkg_yolox", None)
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/edge_ai"
DRIVE_DATA = f"{DRIVE_ROOT}/data"
DRIVE_WEIGHTS = f"{DRIVE_ROOT}/weights"
ROUND = "nas_rank1"   # gerçek NAS, sıfırdan 60 ep sıralama referansı (Karar 6)
SUPER = f"{DRIVE_ROOT}/results/nas_super1"; SEARCH = f"{DRIVE_ROOT}/results/nas_search1"
DRIVE_RESULTS = f"{DRIVE_ROOT}/results/{ROUND}"
for d in (DRIVE_DATA, DRIVE_WEIGHTS, DRIVE_RESULTS):
    os.makedirs(d, exist_ok=True)

import hashlib, urllib.request, shutil
WEIGHT_URL = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth"
w_drive = f"{DRIVE_WEIGHTS}/yolox_nano.pth"
if not os.path.exists(w_drive):
    urllib.request.urlretrieve(WEIGHT_URL, w_drive)
WEIGHTS = "/content/yolox_nano.pth"
shutil.copy(w_drive, WEIGHTS)

h = hashlib.sha256()
with open(WEIGHTS, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
env["weights_url"] = WEIGHT_URL
env["weights_sha256"] = h.hexdigest()
env["weights_bytes"] = os.path.getsize(WEIGHTS)
print(env["weights_sha256"], env["weights_bytes"])

In [ ]:
# 5 · Veri: tam COCO train2017 + val2017 (Colab yereline)
# train2017 görüntüleri Drive'da yok: Drive'da `data/train2017.zip` varsa oradan, yoksa resmi COCO sunucusundan
# Colab yereline indirilir (≈18 GB). Drive kotası için zip varsayılan olarak Drive'a KAYDEDİLMEZ; her yeni
# runtime'da yeniden indirilir. Anotasyonlar ve val2017 P0'dan beri Drive'da.
# Kesinti testi (hücre 7) için 10k alt kümenin yalnız anotasyon json'u açılır (görselleri train2017'de zaten var).
import zipfile, shutil, time
COCO_LOCAL = "/content/datasets/COCO"
ANN_ZIP = f"{DRIVE_DATA}/annotations_trainval2017.zip"
VAL_ZIP = f"{DRIVE_DATA}/val2017.zip"
SUBSET_ZIP = f"{DRIVE_DATA}/coco_train2017_10k_stratified.zip"
TRAIN_ZIP_DRIVE = f"{DRIVE_DATA}/train2017.zip"
TRAIN_ZIP_URL = "http://images.cocodataset.org/zips/train2017.zip"
SAVE_TRAIN_ZIP_TO_DRIVE = False   # True: ilk indirmede Drive'a da kopyalar (≈18 GB kota)
N_TRAIN = 118_287
for z in (ANN_ZIP, VAL_ZIP, SUBSET_ZIP):
    assert os.path.exists(z), f"{z} yok: önce 00 pilot notebook'unun hücre 5 ve 9'u"
t0 = time.time()
os.makedirs(f"{COCO_LOCAL}/annotations", exist_ok=True)
with zipfile.ZipFile(ANN_ZIP) as z:
    for name in ("annotations/instances_train2017.json", "annotations/instances_val2017.json"):
        if not os.path.exists(f"{COCO_LOCAL}/{name}"):
            z.extract(name, COCO_LOCAL)
with zipfile.ZipFile(SUBSET_ZIP) as z:
    if not os.path.exists(f"{COCO_LOCAL}/annotations/instances_train2017_10k.json"):
        z.extract("annotations/instances_train2017_10k.json", COCO_LOCAL)
if not os.path.isdir(f"{COCO_LOCAL}/val2017"):
    shutil.copy(VAL_ZIP, "/content/val2017.zip")
    with zipfile.ZipFile("/content/val2017.zip") as z:
        z.extractall(COCO_LOCAL)

n_local = len(os.listdir(f"{COCO_LOCAL}/train2017")) if os.path.isdir(f"{COCO_LOCAL}/train2017") else 0
if n_local != N_TRAIN:
    local_zip = "/content/train2017.zip"
    if os.path.exists(TRAIN_ZIP_DRIVE):
        print("train2017.zip Drive'dan kopyalanıyor…", flush=True)
        shutil.copy(TRAIN_ZIP_DRIVE, local_zip)
        env["train_zip_source"] = "drive"
    else:
        print("train2017.zip resmi COCO sunucusundan indiriliyor (≈18 GB)…", flush=True)
        subprocess.run(["wget", "-q", "--show-progress", "--progress=dot:giga", "-O", local_zip + ".part",
                        TRAIN_ZIP_URL], check=True)
        os.replace(local_zip + ".part", local_zip)
        env["train_zip_source"] = TRAIN_ZIP_URL
    env["train_zip_bytes"] = os.path.getsize(local_zip)
    env["train_zip_sha256"] = ns.sha256(local_zip)
    if SAVE_TRAIN_ZIP_TO_DRIVE and not os.path.exists(TRAIN_ZIP_DRIVE):
        shutil.copy(local_zip, TRAIN_ZIP_DRIVE + ".tmp")
        os.replace(TRAIN_ZIP_DRIVE + ".tmp", TRAIN_ZIP_DRIVE)
    print("açılıyor…", flush=True)
    subprocess.run(["unzip", "-q", "-o", local_zip, "-d", COCO_LOCAL], check=True)
    os.remove(local_zip)
env["train_images"] = len(os.listdir(f"{COCO_LOCAL}/train2017"))
env["val_images"] = len(os.listdir(f"{COCO_LOCAL}/val2017"))
env["train_ann_sha256"] = ns.sha256(f"{COCO_LOCAL}/annotations/instances_train2017.json")
env["data_prep_s"] = round(time.time() - t0, 1)
env["disk_free_after_data"] = sh("df -h /content | tail -1")
print({k: env.get(k) for k in ("train_images", "val_images", "train_zip_source", "train_zip_sha256",
                               "train_ann_sha256", "data_prep_s", "disk_free_after_data")})
assert env["train_images"] == N_TRAIN and env["val_images"] == 5000

In [ ]:
# 6 · Veri ayrımları (03 ile birebir aynı olmalı)
SPLIT_SEED = 2026
sm_local = f"{COCO_LOCAL}/annotations/nas_splits_manifest.json"
if not os.path.exists(sm_local):
    r = subprocess.run(["python", f"{WS}/scripts/make_splits.py", "--coco", COCO_LOCAL, "--seed", str(SPLIT_SEED)],
                       capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:]); assert r.returncode == 0
sm = json.load(open(sm_local))
sm_super = ns.read_json(f"{SUPER}/splits_manifest.json")
assert sm["ids_sha256"] == sm_super["ids_sha256"], "ayrım 03'tekinden farklı"
print("ayrım 03 ile aynı:", sm["ids_sha256"]["A"][:12])
# MÜHÜR: instances_val2017_B.json hiçbir yükleyiciye verilmez.

In [ ]:
# 7 · Adaylar + öz-denetim + GPU kesinti testi (sıfırdan trainer; 10k, 2 epoch; ≈ 15 dk) + maliyet ölçümü
import torch, re, time, signal
from ckpt_store import CheckpointStore
cands = ns.read_json(f"{SEARCH}/candidates.json")
assert cands and all(c["config_id"] for c in cands["selected"]), "03b aday listesi eksik/boş seçim var"
RUNS = []
for i, c in enumerate(sorted(cands["selected"], key=lambda c: (c["group"] != "nano", c["group"], c["lut_ratio"])), 1):
    RUNS.append({"run_id": f"r{i:02d}_{c['group']}", "config_id": c["config_id"], "group": c["group"],
                 "res": sn.parse_config_id(c["config_id"])["res"], "lut_ms": c["lut_ms"],
                 "supernet_A": cands["a_scores"][c["config_id"]]["map5095"]})
NANO = next(r for r in RUNS if r["group"] == "nano")
assert len(RUNS) == len({r["config_id"] for r in RUNS}) == cands["notes"]["n_selected"] and len(RUNS) >= 2
print("03b notları:", cands["notes"])
for r in RUNS:
    print(r["run_id"], r["config_id"], "LUT", r["lut_ms"], "süper ağ A", r["supernet_A"])
for script, extra in (("check_resume.py", []), ("check_trainer_cpu.py", ["--exp", "scratch"])):
    rr = subprocess.run(["python", f"{WS}/scripts/{script}"] + extra, capture_output=True, text=True, cwd=WS)
    print(rr.stdout[-600:]); assert "GEÇTİ" in rr.stdout.splitlines()[-1], rr.stderr[-3000:]

BATCH, FP16, CU_PER_HOUR = 64, True, 1.54
WORKERS = os.cpu_count()
OUT_ROOT = f"{WS}/YOLOX_outputs/{ROUND}"
MAX_EPOCH, WARMUP, NO_AUG = 60, 1, 3          # orantılı kısaltılmış reçete (Furkan, 2026-10-17): 5/300, 15/300
SEED_INIT, SEED_DATA, CKPT_EVERY = 0, 0, 300
RECIPE_OPTS = ["max_epoch", str(MAX_EPOCH), "warmup_epochs", str(WARMUP), "no_aug_epochs", str(NO_AUG),
               "nas_seed_init", str(SEED_INIT), "nas_seed_data", str(SEED_DATA)]

def run_train(opts, tag, env_extra=None, kill_at=None):
    cmd = ["python", "-u", "YOLOX/tools/train.py", "-f", EXP_FILE, "-d", "1", "-b", str(BATCH), "--fp16",
           "-expn", tag] + opts
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=WS, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env={**os.environ, **(env_extra or {})})
    lines, last = [], 0
    for line in proc.stdout:
        lines.append(line)
        m = re.search(r"epoch: (\d+)/\d+, iter: (\d+)/(\d+)", line)
        if (m and time.time() - last > 120) or re.search(r"devam:|\[test\]|Traceback|Error", line):
            print(f"[{tag} {time.time() - t0:6.0f}s] {line.strip()[-150:]}", flush=True); last = time.time()
        if kill_at is not None and m and (int(m.group(1)) - 1) * int(m.group(3)) + int(m.group(2)) >= kill_at:
            proc.send_signal(signal.SIGKILL); print(f"[{tag}] SIGKILL @ ≥{kill_at}", flush=True)
    proc.wait()
    return proc.returncode, "".join(lines), time.time() - t0

if not os.path.exists(f"{DRIVE_RESULTS}/resume_test.json"):
    T = f"{DRIVE_RESULTS}/_test"
    shutil.rmtree(T, ignore_errors=True)
    arch = next(r for r in RUNS if r["group"] == "stratified")["config_id"]
    common = ["nas_arch", arch, "output_dir", f"{OUT_ROOT}/_test", "data_num_workers", str(WORKERS),
              "train_ann", "instances_train2017_10k.json", "max_epoch", "2", "warmup_epochs", "1", "no_aug_epochs", "0",
              "nas_ckpt_every_iters", "50", "nas_deterministic", "True", "nas_trace_iters", "312"]
    def o(n):
        os.makedirs(f"{T}/{n}", exist_ok=True)
        return common + ["nas_ckpt_dir", f"{T}/{n}/ckpt", "nas_trace_path", f"{T}/{n}/trace.jsonl", "nas_log_dir", f"{T}/{n}"]
    rc, lg, wall_ref = run_train(o("ref"), "_test/ref"); assert rc == 0, lg[-3000:]
    rc1, lg1, _ = run_train(o("test"), "_test/t1", env_extra={"NAS_TEST_KILL_AT_SAVE": "100"}); assert rc1 == 137, lg1[-2000:]
    rc2, lg2, _ = run_train(o("test"), "_test/t2", kill_at=230); assert rc2 == -9 and "ilerleme 50)" in lg2, lg2[-2000:]
    rc3, lg3, _ = run_train(o("test"), "_test/t3"); assert rc3 == 0 and "ilerleme 206)" in lg3, lg3[-2000:]
    cmp_ = nd.compare_traces(f"{T}/ref/trace.jsonl", f"{T}/test/trace.jsonl")
    a = CheckpointStore(f"{T}/ref/ckpt", "/tmp/ck_a").latest()[0]; b = CheckpointStore(f"{T}/test/ckpt", "/tmp/ck_b").latest()[0]
    wdiff = max((a["model_raw"][k].float() - b["model_raw"][k].float()).abs().max().item() for k in a["model_raw"])
    it = [float(x) for x in re.findall(r"iter_time: ([\d.]+)", lg)]
    rt = {**cmp_, "final_weight_max_abs_diff": wdiff, "arch": arch, "rc": [rc1, rc2, rc3],
          "ref_iter_s_last_quarter_deterministic": sorted(it[3 * len(it) // 4:])[len(it[3 * len(it) // 4:]) // 2],
          "tested_utc": ns.now_utc()}
    print(json.dumps(rt, indent=1, default=str))
    assert rt["n_exact_mismatches"] == 0 and not rt["missing_in_test"]
    ns.write_json(f"{DRIVE_RESULTS}/resume_test.json", rt)
else:
    print("kesinti testi kayıtlı:", {k: ns.read_json(f"{DRIVE_RESULTS}/resume_test.json")[k]
                                    for k in ("n_exact_mismatches", "final_weight_max_abs_diff")})

In [ ]:
# 8 · Dondurulmuş manifest: adaylar, tarif, değerlendirme, sıralama analizi, finalist kuralı, cihaz protokolü
from yolox.exp import get_exp
for _m in ("scratch_subnet", "supernet", "nano_macro"):
    sys.modules.pop(_m, None)
_e = get_exp(EXP_FILE, None)
_e.merge(["nas_arch", NANO["config_id"]] + RECIPE_OPTS)
RECIPE_KEYS = ("max_epoch", "warmup_epochs", "warmup_lr", "no_aug_epochs", "basic_lr_per_img", "min_lr_ratio",
               "scheduler", "weight_decay", "momentum", "ema", "input_size", "random_size", "mosaic_prob",
               "mosaic_scale", "enable_mixup", "mixup_prob", "hsv_prob", "flip_prob", "degrees", "translate", "shear",
               "train_ann", "val_ann", "nmsthre", "nas_seed_init", "nas_seed_data")
FINGERPRINT = ("round", "runs", "recipe", "eval_protocol", "rank_analysis", "selection_rule", "latency_protocol",
               "candidates_sha256", "exp_file_sha256", "train_ann_sha256", "splits", "yolox_commit")
manifest = ns.ensure_manifest(f"{DRIVE_RESULTS}/manifest.json", {
    "round": ROUND, "notebook": "notebooks/03c_gercek_nas_siralama_referansi.ipynb",
    "runs": RUNS, "candidates_sha256": ns.sha256(f"{SEARCH}/candidates.json"),
    "recipe": {**{k: getattr(_e, k, None) for k in RECIPE_KEYS}, "batch": BATCH, "fp16": FP16, "init": "sıfırdan (YOLOX)",
               "pretrained": None, "ckpt_every_iters": CKPT_EVERY, "train_images": 118_287,
               "recipe_note": "orantılı kısaltılmış reçete (5/300, 15/300; Furkan 2026-10-17); 300 ep sıralamasını garanti etmez"},
    "eval_protocol": {"split": "val2017 A (2500)", "checkpoint": "son epoch EMA", "precision": "fp32", "conf": 0.001,
                      "nms": 0.65, "resolution": "adayın kendi çözünürlüğü", "ap_scale": "0–100"},
    "rank_analysis": {"module": "scripts/rank_select.py::rank_analysis", "supernet_side": "03b tam val2017-A, BN kalibreli",
                      "scratch_side": "bu tur val2017-A", "bootstrap": "mimari, grup içi", "n_boot": rs.BOOTSTRAP_N,
                      "pair_tolerance": "|T_i − T_j| ≤ 0,05 · T_N (adayların birbirine göre %5'i değil); çift sayısı raporlanır; "
                                        "mimari paylaşan çiftler bağımsız değil", "scope": "panele koşullu; kısmi eğitim referansı"},
    "selection_rule": {"accuracy": "AP_A,60(aday) ≥ AP_A,60(Nano@416) − 1,0 (mutlak, 0–100)",
                       "speed": "asgari gecikme azalması: T_aday ≤ (1 − δ) · T_N, T = R=3 benchmark tekrarının medyanlarının medyanı; "
                                "δ = max(0,01; 3 · CV_N), CV_N = std/mean(Nano tekrar medyanları), aday sonuçlarından önce yalnız "
                                "Nano kaydından; pilotla belirlenen mühendislik güvenlik payı (istatistiksel anlamlılık değil); "
                                "adayların kendi tekrar CV'leri raporlanır",
                       "pareto_100_note": "1,00 · T_N hedefi yalnız aday toplar; finalist için iki koşul ayrıca gerekir",
                       "choose": "uygunlar arasında en düşük medyan gecikme", "none": "eşik gevşetilmez",
                       "module": "scripts/rank_select.py::finalist, ::speed_margin"},
    "latency_protocol": {"script": "scripts/n4_latency.py", "decision": "Karar 3: 100 ısınma + 60 sn × 3, ORT CPU 4 "
                         "thread, ORT_ENABLE_ALL, yalnız session.run, gerçek görüntü girdisi (val2017 000000000139)",
                         "device": "jetson_orin_nano_cpu (ön; RPi 5 gelince rpi5 ile tekrar, seçim ölçütü RPi)",
                         "p95": "raporlanır"},
    "exp_file_sha256": ns.sha256(EXP_FILE), "train_ann_sha256": env["train_ann_sha256"],
    "splits": {"ids_sha256": sm["ids_sha256"], "sealed": "val2017 B açılmaz"},
    "yolox_commit": env["yolox_commit"], "repo_commit_at_creation": env["repo_commit"],
    "resume_test": ns.read_json(f"{DRIVE_RESULTS}/resume_test.json"), "cu_per_hour": CU_PER_HOUR, "env_at_creation": env,
}, keys=FINGERPRINT)
del _e
print(json.dumps({k: manifest[k] for k in ("round", "created_utc", "recipe")}, indent=1, default=str)[:3000])

In [ ]:
# 9 · Eğitim + val-A değerlendirmesi (UZUN KOŞU ≈ 7 gün — Furkan'ın başlatma onayıyla). Kaldığı yerden devam.
# Sıra: Nano önce (referans), sonra diğerleri. Her koşu bitince hemen değerlendirilir (sonuçlar birikerek gelir).
IPE = 118_287 // BATCH
def run_dir(r):
    return f"{DRIVE_RESULTS}/runs/{r['run_id']}"

def train_eval(r):
    st = ns.get_status(DRIVE_RESULTS, r["run_id"])
    if st["status"] == "evaluated":
        return st
    d = run_dir(r); os.makedirs(d, exist_ok=True)
    store = CheckpointStore(f"{d}/ckpt", f"{OUT_ROOT}/{r['run_id']}/ck_local", log=print)
    recs = store.records() if os.path.isdir(f"{d}/ckpt") else []
    p = recs[-1]["progress"] if recs else None
    if p is None or p < MAX_EPOCH * IPE:
        st = ns.set_status(DRIVE_RESULTS, r["run_id"], st, status="training", attempts=st["attempts"] + [
            {"start_utc": ns.now_utc(), "resume_from_progress": p, "gpu": env["gpu"], "repo_commit": env["repo_commit"]}])
        rc, lg, wall = run_train(["nas_arch", r["config_id"], "nas_ckpt_dir", f"{d}/ckpt", "nas_log_dir", d,
                                  "output_dir", OUT_ROOT, "data_num_workers", str(WORKERS),
                                  "nas_ckpt_every_iters", str(CKPT_EVERY), "nas_snapshot_epochs", "30"] + RECIPE_OPTS,
                                 r["run_id"])
        st["attempts"][-1].update(end_utc=ns.now_utc(), wall_s=round(wall, 1), rc=rc)
        st = ns.set_status(DRIVE_RESULTS, r["run_id"], st)
        assert rc == 0, f"{r['run_id']} çıkış kodu {rc}: hücreyi yeniden çalıştır (kayıttan devam)\n{lg[-2000:]}"
    ns.check_train_log(open(f"{d}/train_log.txt").read(), MAX_EPOCH, NO_AUG)
    state, rec = store.find(MAX_EPOCH * IPE)
    assert state is not None
    ck = f"{d}/ckpt/{rec['file']}"
    txt, wall_e = ns.stream(["python", "-u", "YOLOX/tools/eval.py", "-f", EXP_FILE, "-c", ck, "-b", "64", "-d", "1",
                             "--conf", "0.001", "--tsize", str(r["res"]), "-expn", f"{r['run_id']}_evalA",
                             "nas_arch", r["config_id"], "output_dir", f"{OUT_ROOT}_eval"], WS, ns.SHOW_EVAL, r["run_id"])
    ev = ns.parse_eval(txt)
    assert ev["map5095"] is not None
    open(f"{d}/eval_A_log.txt.tmp", "w").write(txt); os.replace(f"{d}/eval_A_log.txt.tmp", f"{d}/eval_A_log.txt")
    train_s = round(sum(a.get("wall_s") or 0 for a in st["attempts"]), 1)
    return ns.set_status(DRIVE_RESULTS, r["run_id"], st, status="evaluated", eval_A={**ev, "eval_s": round(wall_e, 1)},
                         ckpt=rec, train_s=train_s, evaluated_utc=ns.now_utc())

for r in RUNS:
    st = train_eval(r)
    print(f"✔ {r['run_id']} {r['config_id']}: A mAP {st['eval_A']['map5095']} · eğitim {st['train_s']} sn", flush=True)

In [ ]:
# 10 · Sonuçlar + sıralama analizi (Kendall τ, grup korumalı bootstrap, benzer gecikme çiftleri)
import csv
rows = []
for r in RUNS:
    st = ns.get_status(DRIVE_RESULTS, r["run_id"])
    if st["status"] == "evaluated":
        rows.append({**r, "scratch": st["eval_A"]["map5095"], "supernet": r["supernet_A"], "latency_ms": r["lut_ms"],
                     "ap50": st["eval_A"]["ap50"], "train_s": st["train_s"]})
with open(f"{DRIVE_RESULTS}/results.csv.tmp", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["run_id", "group", "config_id", "res", "lut_ms", "supernet", "scratch", "ap50",
                                      "train_s"], extrasaction="ignore"); w.writeheader(); w.writerows(rows)
os.replace(f"{DRIVE_RESULTS}/results.csv.tmp", f"{DRIVE_RESULTS}/results.csv")
for x in rows:
    print(f"{x['run_id']:16s} LUT {x['lut_ms']:7.2f} · süper ağ A {x['supernet']:5.2f} · sıfırdan 60 ep A {x['scratch']:5.2f}")
if len(rows) == len(RUNS):
    ra = rs.rank_analysis(rows, NANO["lut_ms"])
    ns.write_json(f"{DRIVE_RESULTS}/rank_analysis.json", {**ra, "rows": rows, "written_utc": ns.now_utc()})
    print(json.dumps(ra, indent=1))

In [ ]:
# 11 · Cihaz gecikmesi paketi (11 aday, sıfırdan 60 ep ağırlıkları, ham çıktı ONNX opset 13) + eşdeğerlik
# Jetson'da: python3 scripts/n4_latency.py --pkg <paket> --device jetson_orin_nano_cpu --out <çıkış> (önce --smoke)
import numpy as np, onnxruntime as ort
PKG = f"{DRIVE_RESULTS}/latency_pkg"
os.makedirs(f"{PKG}/inputs", exist_ok=True)
inputs = nx.write_latency_inputs(f"{COCO_LOCAL}/val2017/{nx.LATENCY_IMAGE}", f"{PKG}/inputs",
                                 sorted({r["res"] for r in RUNS}))
cand_rows, sums = [], {f"inputs/{i['file']}": i["sha256"] for i in inputs}
for r in RUNS:
    st = ns.get_status(DRIVE_RESULTS, r["run_id"]); assert st["status"] == "evaluated"
    for _m in ("scratch_subnet",):
        sys.modules.pop(_m, None)
    e_ = get_exp(EXP_FILE, None); e_.merge(["nas_arch", r["config_id"]])
    m = e_.get_model().eval()
    m.load_state_dict(torch.load(f"{run_dir(r)}/ckpt/{st['ckpt']['file']}", map_location="cpu", weights_only=False)["model"], strict=True)
    m.head.decode_in_inference = False
    path = f"{PKG}/{r['run_id']}.onnx"
    if not os.path.exists(path):
        nx.export_onnx(m, r["res"], path)
    x = np.load(f"{PKG}/inputs/{nx.latency_input_name(r['res'])}")
    o = ort.InferenceSession(path, providers=["CPUExecutionProvider"]).run(None, {nx.INPUT_NAME: x})[0][0]
    with torch.no_grad():
        ref = m(torch.from_numpy(x))[0].numpy()
    dd = nx.decoded_diff(ref, o, r["res"]); assert dd["pass"] and dd["n_scored_anchors"] > 0, (r["run_id"], dd)
    sums[f"{r['run_id']}.onnx"] = ns.sha256(path)
    cand_rows.append({"candidate_id": r["run_id"], "tier": r["group"], "selectable": True, "arch_id": r["config_id"],
                      "resolution": r["res"], "map5095": st["eval_A"]["map5095"]})
    print(r["run_id"], "ONNX eşdeğer:", {k: dd[k] for k in ("max_box_px", "max_score_diff", "n_scored_anchors")})
nx.write_csv(f"{PKG}/n4_candidates.csv", cand_rows, list(cand_rows[0].keys()))
with open(f"{PKG}/SHA256SUMS.tmp", "w") as f:
    for k, v in sorted(sums.items()):
        f.write(f"{v}  {k}\n")
os.replace(f"{PKG}/SHA256SUMS.tmp", f"{PKG}/SHA256SUMS")
shutil.make_archive(f"{DRIVE_RESULTS}/nas_rank1_latency_pkg", "zip", DRIVE_RESULTS, "latency_pkg")
print("paket:", f"{DRIVE_RESULTS}/nas_rank1_latency_pkg.zip", ns.sha256(f"{DRIVE_RESULTS}/nas_rank1_latency_pkg.zip"))

In [ ]:
# 12 · Finalist (önceden kayıtlı kural). Girdi: Jetson ölçüm çıktısı Drive'a → {DRIVE_RESULTS}/latency/jetson_orin_nano_cpu/
# Önce YALNIZ Nano kaydından asgari hızlanma payı hesaplanıp dondurulur; adayların gecikmesi ondan sonra okunur.
LAT_DIR = f"{DRIVE_RESULTS}/latency/jetson_orin_nano_cpu"
mpath = f"{DRIVE_RESULTS}/speed_margin.json"
if not os.path.exists(mpath):
    nano_rec = ns.read_json(f"{LAT_DIR}/runs/{NANO['run_id']}.json")
    assert nano_rec, "Nano ölçüm kaydı yok"
    sm_ = rs.speed_margin([rep["median_ms"] for rep in nano_rec["repeats"]])
    ns.write_json(mpath, {**sm_, "from": f"{NANO['run_id']}.json", "frozen_utc": ns.now_utc()})
margin = ns.read_json(mpath)["margin"]
fpath = f"{DRIVE_RESULTS}/finalist.json"
if not os.path.exists(fpath):
    import statistics as stt
    frows = []
    for r in RUNS:
        rec = ns.read_json(f"{LAT_DIR}/runs/{r['run_id']}.json"); assert rec, f"{r['run_id']} ölçülmemiş"
        meds = [rep["median_ms"] for rep in rec["repeats"]]
        frows.append({"config_id": r["config_id"], "run_id": r["run_id"],
                      "ap_a60": ns.get_status(DRIVE_RESULTS, r["run_id"])["eval_A"]["map5095"],
                      "lat_median_ms": stt.median(meds), "lat_p95_ms": rec["summary"].get("latency_ms_p95"),
                      "rep_medians_ms": meds})
    fin = rs.finalist(frows, NANO["config_id"], margin)
    ns.write_json(fpath, {**fin, "rows": frows, "decided_utc": ns.now_utc(), "device": "jetson_orin_nano_cpu"})
fin = ns.read_json(fpath)
print("KARAR:", fin["verdict"], "| finalist:", fin["finalist"], "| uygunlar:", fin["eligible"], "| pay:", margin)